# 02 · Анализ spread
Цель — ответить на четыре вопроса и **обосновать порог алерта распределением, а не с потолка**:
1. Какой spread типичен?
2. В какие часы он шире?
3. Какая монета самая «расходящаяся»?
4. Какой порог даст разумное число алертов?

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import matplotlib.pyplot as plt
from storage.db import connect

def q(sql, params=()):
    """Запрос к базе -> DataFrame. Соединение только на чтение и сразу закрывается."""
    with connect(read_only=True) as con:
        return con.execute(sql, list(params)).df()

In [ ]:
s = q("SELECT * FROM spreads ORDER BY ts")
s["hour"] = s.ts.dt.hour
print(len(s), "строк;", s.ts.min(), "—", s.ts.max())

## 1. Распределение spread

In [ ]:
ax = s.spread_bp.clip(upper=s.spread_bp.quantile(0.995)).hist(bins=60, figsize=(9, 3.5))
ax.set_xlabel("spread, б.п. за 8 ч"); ax.set_ylabel("число наблюдений")
for p in (0.5, 0.95, 0.99):
    ax.axvline(s.spread_bp.quantile(p), linestyle="--", color="black", linewidth=1)
s.spread_bp.describe(percentiles=[.5, .9, .95, .99]).round(3)

## 2. Spread по часам суток (UTC)
То же самое одним SQL-запросом — оконные и агрегатные функции на временном ряду.

In [ ]:
by_hour = q("""SELECT hour(ts) AS hour, median(spread_bp) AS median_bp, quantile_cont(spread_bp, 0.95) AS p95_bp
               FROM spreads GROUP BY 1 ORDER BY 1""")
by_hour.plot(x="hour", y=["median_bp", "p95_bp"], figsize=(9, 3.5), xlabel="час (UTC)", ylabel="б.п.")
by_hour.T.round(2)

## 3. Какая монета самая «расходящаяся» и кто чаще дороже

In [ ]:
q("""SELECT symbol, round(median(spread_bp), 3) AS median_bp, round(quantile_cont(spread_bp, 0.95), 3) AS p95_bp,
            round(max(spread_bp), 2) AS max_bp, mode(exchange_hi) AS usually_most_expensive
     FROM spreads GROUP BY symbol ORDER BY p95_bp DESC""")

In [ ]:
q("""SELECT exchange_hi, exchange_lo, count(*) AS n,
            round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
     FROM spreads GROUP BY ALL ORDER BY n DESC""")

## 4. Топ эпизодов
Эпизод — подряд идущие слоты, где spread выше порога. Классический приём «gaps and islands»: номер острова = накопленная сумма флагов «начался новый».

In [ ]:
THRESHOLD = 1.5
episodes = q("""
WITH flagged AS (
    SELECT symbol, ts, spread_bp, spread_bp >= ? AS above,
           lag(spread_bp >= ?, 1, false) OVER (PARTITION BY symbol ORDER BY ts) AS prev_above
    FROM spreads),
islands AS (
    SELECT *, sum(CASE WHEN above AND NOT prev_above THEN 1 ELSE 0 END)
                  OVER (PARTITION BY symbol ORDER BY ts) AS episode
    FROM flagged)
SELECT symbol, min(ts) AS started, max(ts) AS ended, count(*) * 5 AS minutes, round(max(spread_bp), 2) AS peak_bp
FROM islands WHERE above GROUP BY symbol, episode ORDER BY minutes DESC""", (THRESHOLD, THRESHOLD))
print("эпизодов:", len(episodes), "| из них дольше 15 мин:", (episodes.minutes >= 15).sum())
episodes.head(10)

## 5. Выбор порога
Сколько алертов в сутки получится при разных порогах (с учётом правила «держится ≥ 15 минут»)?

In [ ]:
days = max((s.ts.max() - s.ts.min()).total_seconds() / 86400, 1e-9)
rows = []
for thr in (0.5, 0.75, 1.0, 1.5, 2.0, 3.0):
    above = s.assign(above=s.spread_bp >= thr).sort_values(["symbol", "ts"])
    start = above.above & ~above.groupby("symbol").above.shift(fill_value=False)
    lengths = above[above.above].groupby([above.symbol, start.cumsum()]).size() * 5
    rows.append({"порог, б.п.": thr, "доля времени выше, %": round(above.above.mean() * 100, 2),
                 "эпизодов в сутки": round(len(lengths) / days, 1),
                 "алертов в сутки (≥15 мин)": round((lengths >= 15).sum() / days, 1)})
pd.DataFrame(rows)

## Выводы (заполни сама)
- Типичный spread: … б.п. (медиана), p95 — …
- Шире всего в … часов UTC, потому что …
- Самая «расходящаяся» монета: …
- Порог алерта: … б.п., потому что это даёт … алертов в сутки.

Выбранный порог впиши в `config.yaml` → `alerts.spread_bp` или задай командой `/alerts spread_bp …`.

## Задание
Повтори шаги 1 и 5 для z-score объёма (`analysis.anomalies.volume_zscores` считает только последнюю свечу — напиши запрос для всей истории). У объёма тяжёлый хвост: проверь, сколько алертов в сутки даёт порог 4σ, и подбери свой.